<a href="https://colab.research.google.com/github/ronggobp/Machine-Learning-Course-2026/blob/main/notebooks/week-06-unsupervised-learning/06_KMeans_PCA_Segmentation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Setup, W&B & Load Mall Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import wandb

# Inisialisasi W&B
run = wandb.init(project="customer-segmentation", name="kmeans-pca-baseline")

# Tautan Dataset yang Lebih Stabil
url = "https://raw.githubusercontent.com/tirthajyoti/Machine-Learning-with-Python/master/Datasets/Mall_Customers.csv"
df = pd.read_csv(url)

# Kita hanya gunakan fitur Annual Income dan Spending Score untuk latihan awal
X = df.iloc[:, [3, 4]].values

print(f"Dataset Berhasil Dimuat! Shape: {df.shape}")
df.head()

Mencari K Optimal (Elbow Method)

In [ ]:
wcss = []
for i in range(1, 11):
    kmeans = KMeans(n_clusters=i, init='k-means++', random_state=42)
    kmeans.fit(X)
    wcss.append(kmeans.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(range(1, 11), wcss, marker='o', linestyle='--')
plt.title('Elbow Method untuk Mencari K Optimal')
plt.xlabel('Jumlah Clusters (k)')
plt.ylabel('WCSS')
plt.show()

# Log WCSS ke W&B untuk melihat "siku" grafik
wandb.log({"WCSS_Curve": wandb.Image(plt)})

Training K-Means & Visualisasi Cluster

In [ ]:
# Berdasarkan Elbow, k=5 biasanya optimal untuk dataset ini
kmeans = KMeans(n_clusters=5, init='k-means++', random_state=42)
y_kmeans = kmeans.fit_predict(X)

# Visualisasi Kelompok Pelanggan
plt.figure(figsize=(10, 7))
sns.scatterplot(x=X[y_kmeans == 0, 0], y=X[y_kmeans == 0, 1], s=100, label='Cluster 1 (Hemat)')
sns.scatterplot(x=X[y_kmeans == 1, 0], y=X[y_kmeans == 1, 1], s=100, label='Cluster 2 (Rata-rata)')
sns.scatterplot(x=X[y_kmeans == 2, 0], y=X[y_kmeans == 2, 1], s=100, label='Cluster 3 (Sultan/Target)')
sns.scatterplot(x=X[y_kmeans == 3, 0], y=X[y_kmeans == 3, 1], s=100, label='Cluster 4 (Boros)')
sns.scatterplot(x=X[y_kmeans == 4, 0], y=X[y_kmeans == 4, 1], s=100, label='Cluster 5 (Hati-hati)')
plt.scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1], s=300, c='black', label='Centroids', marker='X')
plt.title('Segmentasi Pelanggan Mall')
plt.xlabel('Annual Income (k$)')
plt.ylabel('Spending Score (1-100)')
plt.legend()
plt.show()

Menggunakan PCA untuk Visualisasi Multidimensi

In [ ]:
# Sekarang kita gunakan semua fitur (Age, Income, Spending)
X_full = df.iloc[:, [2, 3, 4]].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_full)

# Reduksi dari 3D menjadi 2D agar bisa digambar
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

kmeans_full = KMeans(n_clusters=5, random_state=42)
clusters_full = kmeans_full.fit_predict(X_pca)

plt.figure(figsize=(8, 6))
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=clusters_full, cmap='viridis')
plt.title('Visualisasi Cluster Hasil Reduksi PCA')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.show()

wandb.finish()